# 16.4 PageRank 与 HITS / PageRank & HITS

**中文**：上一节的中心性都在**无向**图上。但网络很多是**有向**的——网页之间的超链接、论文引用、Twitter 关注。怎么在有向图上衡量"重要性"？**PageRank**(Google 的立身之本)和 **HITS** 是两个里程碑式的答案，它们的核心思想——"被重要的人指向才算重要"——至今贯穿搜索、推荐、反作弊。
**English**: Last section's centralities were on **undirected** graphs. But many networks are **directed** — hyperlinks between web pages, paper citations, Twitter follows. How to measure "importance" on a directed graph? **PageRank** (Google's founding algorithm) and **HITS** are two landmark answers; their core idea — "you are important if important things point to you" — still permeates search, recommendation, and anti-spam.

---

**中文**：**PageRank 的随机冲浪者模型**：想象一个人在网上乱点链接。在任意页面，他以概率 $d$(阻尼系数, 通常 0.85)随机点击当前页的一个出链，以概率 $1-d$ **瞬移(teleport)** 到任意随机页面(防止卡在死胡同)。**一个页面的 PageRank = 长期来看冲浪者停在它上面的概率**。
**English**: **PageRank's random-surfer model**: imagine someone randomly clicking links. On any page, with probability $d$ (damping, usually 0.85) they follow a random out-link of the current page; with probability $1-d$ they **teleport** to a uniformly random page (escaping dead ends). **A page's PageRank = the long-run probability the surfer is on it.**

$$\mathbf r = \frac{1-d}{N}\mathbf 1 + d\,M^\top \mathbf r,\qquad M_{ij}=\frac{A_{ij}}{\text{out-deg}(i)}$$

**中文**：逐项解释：$\mathbf r$ 是各页面 PageRank 向量(和为1)；$N$ 是页面数；$M$ 是**行归一化的转移矩阵**($M_{ij}$=从 $i$ 走到 $j$ 的概率=$i$ 的出链中指向 $j$ 的比例)；$\frac{1-d}{N}\mathbf 1$ 是瞬移项。这是一个**不动点方程**——$\mathbf r$ 是转移矩阵的主特征向量，用**幂迭代(power iteration)** 反复代入即可收敛。**死结(dangling, 无出链)节点**特殊处理：让它瞬移到所有页面(整行设为 $1/N$)。
**English**: Term by term: $\mathbf r$ is the PageRank vector (sums to 1); $N$ = #pages; $M$ is the **row-normalized transition matrix** ($M_{ij}$ = probability of going $i\to j$ = fraction of $i$'s out-links pointing to $j$); $\frac{1-d}{N}\mathbf 1$ is the teleport term. This is a **fixed-point equation** — $\mathbf r$ is the dominant eigenvector of the transition matrix, reached by **power iteration** (repeated substitution). **Dangling nodes** (no out-links) are handled by teleporting everywhere (their row set to $1/N$).

> 💡 **面试速查 / Interview cheat-sheet（★★★ 经典必考）**
> **中文**：**PageRank**=有向图上"被重要页面指向才重要"的随机游走稳态分布；幂迭代解主特征向量；阻尼 $d$=0.85 兼顾"跟随链接"与"瞬移"(也解决周期/不连通/死结)。**关键认知**:PageRank **≠ 入度**——一个来自高 PR 页面的链接，远比一堆垃圾页的链接值钱。**HITS**:每个节点有两个分数——**权威值 authority**(被好的 hub 指向)和**枢纽值 hub**(指向好的 authority),互相加强,迭代 $a=A^\top h,\ h=Aa$。PageRank 全局查询无关、可离线算; HITS 早期按查询子图计算。应用:搜索排序、反欺诈(刷单/僵尸粉)、推荐(物品重要性)、TextRank 关键词抽取。
> **English**: **PageRank** = the stationary distribution of a random walk on a directed graph where "important pages pointing to you make you important"; solved by power iteration (dominant eigenvector); damping $d$=0.85 balances "follow links" vs "teleport" (also fixes periodicity/disconnection/dead-ends). **Key insight**: PageRank **≠ in-degree** — one link from a high-PR page is worth far more than many junk links. **HITS**: each node has two scores — **authority** (pointed to by good hubs) and **hub** (points to good authorities), mutually reinforcing, iterating $a=A^\top h,\ h=Aa$. PageRank is query-independent and precomputable offline; classic HITS was computed per-query on a subgraph. Uses: search ranking, anti-fraud (click farms/bot followers), recommendation (item importance), TextRank keyword extraction.


In [ ]:

# ============================================================
# 数据：一个小型有向"网页图" / a small directed web graph
# 中文：节点=网页, 有向边 i->j 表示"页面 i 链接到页面 j"。我们故意设计让"入度"与"PageRank"分道扬镳。
# English: nodes=pages, directed edge i->j means "page i links to page j". Designed so in-degree ≠ PageRank.
# ============================================================
import networkx as nx, numpy as np, matplotlib.pyplot as plt
np.random.seed(0)
edges=[(0,1),(0,2),(1,2),(2,0),(3,2),(2,3),(4,2),(5,2),(6,2),(2,5),(7,0),(8,0),(9,0)]
G=nx.DiGraph(edges)
nodes=sorted(G.nodes()); N=len(nodes)
A=nx.to_numpy_array(G,nodelist=nodes)                  # A[i,j]=1 表示 i->j / adjacency
print("页面数 / #pages:", N, "| 链接数 / #links:", G.number_of_edges())
print("各页面入度 / in-degree:", dict(G.in_degree()))
print("各页面出度 / out-degree:", dict(G.out_degree()))


**中文**：从零实现 **PageRank 幂迭代**：构造行归一化转移矩阵 $M$(死结行设为均匀)，从均匀分布出发，反复执行 $\mathbf r \leftarrow \frac{1-d}{N} + d\,M^\top\mathbf r$ 直到收敛。
**English**: Implement **PageRank power iteration** from scratch: build the row-normalized transition matrix $M$ (dangling rows set uniform), start from a uniform distribution, and repeat $\mathbf r \leftarrow \frac{1-d}{N} + d\,M^\top\mathbf r$ until convergence.


In [ ]:

# ============================================================
# 从零实现 PageRank 幂迭代 / PageRank power iteration from scratch
# ============================================================
def pagerank(A, d=0.85, iters=100, tol=1e-10):
    N=A.shape[0]; out=A.sum(1)                          # 每个节点出度 / out-degree
    M=np.zeros_like(A)
    for i in range(N):
        M[i]= A[i]/out[i] if out[i]>0 else np.ones(N)/N  # 行归一化; 死结→均匀 / normalize; dangling->uniform
    r=np.ones(N)/N; history=[r.copy()]                  # 从均匀分布出发 / start uniform
    for _ in range(iters):
        r_new=(1-d)/N + d*(M.T@r)                       # 一步幂迭代 / one power-iteration step
        history.append(r_new.copy())
        if np.abs(r_new-r).sum()<tol: r=r_new; break    # 收敛判据 / convergence
        r=r_new
    return r/r.sum(), history

pr, hist = pagerank(A)
pr_nx = nx.pagerank(G, alpha=0.85)                      # NetworkX 对照 / reference
print("从零 PageRank / ours:", {nodes[i]:round(pr[i],4) for i in range(N)})
print("NetworkX 验证 / verify:", all(abs(pr[i]-pr_nx[nodes[i]])<1e-4 for i in range(N)))
print("迭代收敛步数 / converged in:", len(hist)-1, "步/steps")


**中文**：最有洞察的一点——**PageRank ≠ 入度**。看节点 1 和节点 5：它们**入度都是 1**，但 PageRank 差好几倍。为什么？因为节点 5 是被**重要的节点 2**(PageRank 最高)链接的，而节点 1 只被节点 0 链接。**"一票"的分量取决于投票者本身有多重要**——这正是 PageRank 击败"数链接数"的朴素方法、终结早期搜索引擎被堆链接刷榜的关键。
**English**: The key insight — **PageRank ≠ in-degree**. Look at nodes 1 and 5: both have **in-degree 1**, yet their PageRanks differ several-fold. Why? Node 5 is linked by the **important node 2** (highest PageRank), while node 1 is only linked by node 0. **The weight of a "vote" depends on how important the voter is** — exactly how PageRank beat the naive "count the links" approach and ended early search engines being gamed by link stuffing.


In [ ]:

# ============================================================
# PageRank vs 入度：同样入度, 不同重要性 / PageRank vs in-degree
# ============================================================
indeg=dict(G.in_degree())
print(f"{'页面/page':<8}{'入度/in-deg':>10}{'PageRank':>11}{'被谁链接 / linked by':>26}")
for i in nodes:
    src=[u for u,v in G.edges() if v==i]
    print(f"{i:<8}{indeg[i]:>10}{pr[nodes.index(i)]:>11.4f}{str(src):>26}")
print("\n节点1 vs 节点5：入度都=1，但 5 被重要的节点2链接 → PR 高得多")
print(f"  node1 PR={pr[nodes.index(1)]:.4f} (被node0链接)  node5 PR={pr[nodes.index(5)]:.4f} (被重要的node2链接)")


**中文**：**HITS（Hyperlink-Induced Topic Search）** 给每个节点两个分数：
**English**: **HITS (Hyperlink-Induced Topic Search)** gives each node two scores:

**中文**：
- **权威值 authority**：被很多"好枢纽"指向 → 内容权威(如一篇被各大导航页链接的论文)。
- **枢纽值 hub**：指向很多"好权威" → 是优质的导航/索引页(如一个收集了所有权威资源的列表页)。

二者**互相定义、互相加强**：好枢纽指向的页面权威值高；好权威被指向多的页面枢纽值高。迭代更新 $\mathbf a = A^\top\mathbf h$(权威=入边来自的枢纽之和)、$\mathbf h = A\mathbf a$(枢纽=出边指向的权威之和)，每步归一化。
**English**:
- **Authority**: pointed to by many "good hubs" → authoritative content (e.g. a paper linked by major index pages).
- **Hub**: points to many "good authorities" → a high-quality index/navigation page (e.g. a list collecting all the authoritative resources).

The two are **mutually defined and reinforcing**: pages a good hub points to gain authority; pages pointing to good authorities gain hub score. Iterate $\mathbf a = A^\top\mathbf h$ (authority = sum of hubs pointing in), $\mathbf h = A\mathbf a$ (hub = sum of authorities pointed to), normalizing each step.


In [ ]:

# ============================================================
# 从零实现 HITS / HITS from scratch
# ============================================================
def hits(A, iters=100):
    N=A.shape[0]; h=np.ones(N); a=np.ones(N)
    for _ in range(iters):
        a = A.T @ h                                     # 权威 = 指向我的枢纽之和 / authority update
        h = A @ a                                       # 枢纽 = 我指向的权威之和 / hub update
        a/=np.linalg.norm(a) or 1; h/=np.linalg.norm(h) or 1   # 归一化防爆炸 / normalize
    return h, a
h, a = hits(A)
h_nx, a_nx = nx.hits(G, max_iter=1000)
print("权威值 authority (top3):", sorted({nodes[i]:round(a[i],3) for i in range(N)}.items(),key=lambda x:-x[1])[:3])
print("枢纽值 hub       (top3):", sorted({nodes[i]:round(h[i],3) for i in range(N)}.items(),key=lambda x:-x[1])[:3])
# 与 networkx 对比(归一化方式不同, 比较 Top-2 排名; 近零权威的并列顺序无意义) / compare top-2 ranking
rank_ours=sorted(nodes,key=lambda i:-a[nodes.index(i)])
rank_nx=sorted(nodes,key=lambda i:-a_nx[i])
print("权威 Top-2 排名一致 / authority top-2 matches nx:", rank_ours[:2]==rank_nx[:2], rank_ours[:2])


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(17,5))
pos=nx.spring_layout(G,seed=3)
# ① 节点大小=PageRank / node size = PageRank
nx.draw_networkx(G,pos,node_size=[pr[nodes.index(i)]*6000+100 for i in nodes],
                 node_color=[pr[nodes.index(i)] for i in nodes],cmap="YlOrRd",
                 font_size=9,ax=ax[0],edge_color="gray",arrowsize=12)
ax[0].set_title("PageRank(节点越大越重要) / node size = PageRank"); ax[0].axis("off")
# ② PageRank vs 入度 散点 / PR vs in-degree
xs=[indeg[i] for i in nodes]; ys=[pr[nodes.index(i)] for i in nodes]
ax[1].scatter(xs,ys,s=80,c="#4C72B0")
for i in nodes: ax[1].annotate(str(i),(indeg[i],pr[nodes.index(i)]),xytext=(4,3),textcoords="offset points")
ax[1].set_xlabel("入度 in-degree"); ax[1].set_ylabel("PageRank"); ax[1].set_title("PR≠入度(同入度PR可差很多) / PR ≠ in-degree")
# ③ 幂迭代收敛过程 / power-iteration convergence
H=np.array(hist)
for i in [2,0,5,1]: ax[2].plot(H[:,nodes.index(i)],label=f"node {i}")
ax[2].set_title("幂迭代收敛 / power-iteration convergence"); ax[2].set_xlabel("iteration"); ax[2].set_ylabel("PageRank"); ax[2].legend()
plt.tight_layout(); plt.savefig("/tmp/g04_viz.png",dpi=80); plt.show()
print("PageRank 最高 / top page:", nodes[int(np.argmax(pr))], "  收敛很快(几步即稳)/ converges in a few steps")


**中文**：诚实解读：
**English**: Honest takeaways:

**中文**：
1. **PageRank ≠ 入度，这是它的精髓**：节点 1 与 5 入度相同，PageRank 却差几倍——因为重要性会**沿链接传播**。这让"堆一堆垃圾页互链"无法刷高排名(单纯刷入度无效)，是 PageRank 当年碾压竞品的根本。
2. **幂迭代收敛极快**：阻尼 0.85 下通常几十步内收敛(本例几步)。这是因为转移矩阵的第二特征值约为 $d=0.85$，收敛速率由它决定——这也是 $d$ 不取太大的原因之一(太大收敛慢、且易受 spam 链接结构影响)。
3. **PageRank vs HITS**：PageRank **查询无关**、可离线全量预计算、单一分数、对死结/瞬移有鲁棒处理；HITS 给**双分数**(权威/枢纽)、概念上更适合"导航页 vs 内容页"的区分，但原始版本要按查询构子图、计算更重、对 spam 更敏感。工业搜索最终主要沿用 PageRank 思路。
4. **从零实现与 NetworkX 一致**(PageRank 数值吻合、HITS 排名吻合)。

**English**:
1. **PageRank ≠ in-degree — its essence**: nodes 1 and 5 share in-degree but differ several-fold in PageRank because importance **propagates along links**. This makes "a clump of cross-linked junk pages" unable to game rankings (raising raw in-degree doesn't work) — why PageRank crushed competitors.
2. **Power iteration converges fast**: under damping 0.85 it typically converges in tens of steps (a few here). The transition matrix's second eigenvalue is about $d=0.85$, setting the convergence rate — one reason $d$ isn't taken too large (slower convergence, more spam-susceptibility).
3. **PageRank vs HITS**: PageRank is **query-independent**, fully precomputable offline, a single score, with robust dead-end/teleport handling; HITS gives **two scores** (authority/hub), conceptually nicer for "index vs content" pages, but the original computes a per-query subgraph, is heavier, and more spam-sensitive. Industrial search largely followed PageRank.
4. **From-scratch matches NetworkX** (PageRank values agree; HITS ranking agrees).

> 💼 **实战视角 / Practical angle**
> **中文**：PageRank 思想远超搜索:① **推荐**——Personalized PageRank(瞬移只回到种子节点)做"和你相关的重要物品"(Pinterest 的 Pixie、淘宝);② **反欺诈**——识别刷单/僵尸粉(异常互链结构 PageRank 异常);③ **NLP**——TextRank 抽关键词/摘要(把词/句当节点);④ **图数据库**内置算子。**工程**:亿级网页用稀疏矩阵 + 分布式幂迭代(MapReduce/Pregel)。面试金句:*"PageRank 是带瞬移的随机游走稳态, 重要性沿链接传播, 所以高质量入链 ≫ 大量垃圾入链; 它查询无关可离线算, 是它能规模化的关键。"*
> **English**: PageRank's idea goes far beyond search: ① **recommendation** — Personalized PageRank (teleport only to seed nodes) for "important items relevant to you" (Pinterest's Pixie, Taobao); ② **anti-fraud** — spotting click farms / bot followers (anomalous cross-linking → anomalous PageRank); ③ **NLP** — TextRank for keywords/summarization (words/sentences as nodes); ④ built into graph databases. **Engineering**: billions of pages use sparse matrices + distributed power iteration (MapReduce/Pregel). Interview line: *"PageRank is the stationary distribution of a teleporting random walk; importance flows along links, so quality in-links ≫ many junk in-links; being query-independent and offline-computable is what makes it scale."*

---
### 小结 / Summary
- **中文**：PageRank=有向图随机冲浪者稳态分布, 幂迭代求主特征向量, 阻尼 0.85 + 瞬移处理死结/不连通。
- **English**: PageRank = stationary distribution of a directed random surfer; power iteration for the dominant eigenvector; damping 0.85 + teleport handle dead-ends/disconnection.
- **中文**：核心洞察 PageRank≠入度——重要页面的一个链接胜过一堆垃圾链接。
- **English**: Core insight PageRank ≠ in-degree — one link from an important page beats many junk links.
- **中文**：HITS 给权威/枢纽双分数, 互相加强; PageRank 查询无关可离线, 更适合大规模。
- **English**: HITS gives dual authority/hub scores, mutually reinforcing; PageRank is query-independent and offline-friendly, better at scale.
